#### CREDIT PORTFOLIO HEALTH MONITOR
- dbt Modeling & Production Grade Transformations

#### Imports

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

print("Current Directory:", os.getcwd())
print("Ready to configure dbt")

Current Directory: C:\Users\tohiba\Desktop\credit_portfolio_health_monitor\notebooks
Ready to configure dbt


#### Configure dbt profiles.yml

In [2]:
profiles_content = f"""mophones_credit:
  target: dev
  outputs:
    dev:
      type: mysql
      server: {os.getenv('MYSQL_HOST')}
      port: {int(os.getenv('MYSQL_PORT'))}
      database: {os.getenv('MYSQL_DATABASE')}
      username: {os.getenv('MYSQL_USER')}
      password: {os.getenv('MYSQL_PASSWORD')}
      schema: mophones_credit
"""

dbt_dir = os.path.expanduser('~/.dbt')
os.makedirs(dbt_dir, exist_ok=True)

with open(os.path.join(dbt_dir, 'profiles.yml'), 'w') as f:
    f.write(profiles_content)

print("profiles.yml created successfully!")

profiles.yml created successfully!


#### Update dbt_project.yml

In [3]:
config = """name: 'mophones_credit'
version: '1.0.0'
config-version: 2

profile: 'mophones_credit'

model-paths: ["models"]
analysis-paths: ["analyses"]
test-paths: ["tests"]
seed-paths: ["seeds"]
macro-paths: ["macros"]

target-path: "target"
clean-targets:
  - "target"
  - "dbt_packages"

models:
  mophones_credit:
    staging:
      +materialized: view
    intermediate:
      +materialized: view
    marts:
      +materialized: table
"""

with open('../dbt_project/dbt_project.yml', 'w') as f:
    f.write(config)

print("dbt_project.yml updated with proper layering")

dbt_project.yml updated with proper layering


#### Create Model Folders

In [4]:
base = '../dbt_project/models'

folders = [
    f"{base}/staging",
    f"{base}/intermediate",
    f"{base}/marts"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

Created: ../dbt_project/models/staging
Created: ../dbt_project/models/intermediate
Created: ../dbt_project/models/marts


#### Create Staging Models

In [5]:
# 1. stg_customers.sql
stg_customers = """
{{ config(materialized='view') }}

SELECT 
    customer_id,
    full_name,
    phone_number,
    email,
    age,
    gender,
    location,
    income_band,
    credit_score,
    created_at
FROM {{ source('mophones_credit', 'customers') }}
"""

# 2. stg_loans.sql
stg_loans = """
{{ config(materialized='view') }}

SELECT 
    loan_id,
    customer_id,
    disbursed_date,
    principal,
    term_months,
    interest_rate,
    device_model,
    promo_applied
FROM {{ source('mophones_credit', 'loans') }}
"""

# 3. stg_repayments.sql
stg_repayments = """
{{ config(materialized='view') }}

SELECT 
    repayment_id,
    loan_id,
    due_date,
    paid_date,
    amount_due,
    amount_paid,
    days_late,
    CASE 
        WHEN days_late >= 90 THEN '90+'
        WHEN days_late >= 60 THEN '60-89'
        WHEN days_late >= 30 THEN '30-59'
        ELSE 'Current'
    END as delinquency_bucket
FROM {{ source('mophones_credit', 'repayments') }}
"""

# Write the files
base_path = '../dbt_project/models/staging'

with open(f'{base_path}/stg_customers.sql', 'w') as f:
    f.write(stg_customers.strip())

with open(f'{base_path}/stg_loans.sql', 'w') as f:
    f.write(stg_loans.strip())

with open(f'{base_path}/stg_repayments.sql', 'w') as f:
    f.write(stg_repayments.strip())

print("Staging models created successfully!")
print("Files created:")
print("- stg_customers.sql")
print("- stg_loans.sql")
print("- stg_repayments.sql")

Staging models created successfully!
Files created:
- stg_customers.sql
- stg_loans.sql
- stg_repayments.sql


#### Intermediate & Mart Models

In [6]:
base_path = '../dbt_project/models'

# Intermediate Layer
int_loans_repayments = """
{{ config(materialized='view') }}

WITH loan_repayment_summary AS (
    SELECT 
        l.loan_id,
        l.customer_id,
        l.disbursed_date,
        l.principal,
        l.term_months,
        l.device_model,
        l.promo_applied,
        MAX(r.due_date) as latest_due_date,
        SUM(r.amount_due) as total_due,
        SUM(r.amount_paid) as total_paid,
        MAX(r.days_late) as max_days_late,
        COUNT(CASE WHEN r.days_late >= 30 THEN 1 END) as times_30plus_dpd
    FROM {{ ref('stg_loans') }} l
    LEFT JOIN {{ ref('stg_repayments') }} r ON l.loan_id = r.loan_id
    GROUP BY l.loan_id, l.customer_id, l.disbursed_date, l.principal, 
             l.term_months, l.device_model, l.promo_applied
)
SELECT *,
       CASE 
           WHEN max_days_late >= 90 THEN '90+'
           WHEN max_days_late >= 60 THEN '60-89'
           WHEN max_days_late >= 30 THEN '30-59'
           ELSE 'Performing'
       END as current_status
FROM loan_repayment_summary
"""

# Mart Layer (Business Facing)
mart_portfolio_kpis = """
{{ config(materialized='table') }}

SELECT 
    COUNT(DISTINCT customer_id) as total_customers,
    COUNT(loan_id) as total_loans,
    ROUND(SUM(principal), 0) as total_disbursed,
    ROUND(AVG(principal), 0) as avg_loan_size,
    ROUND(SUM(CASE WHEN promo_applied = TRUE THEN principal ELSE 0 END) * 100.0 / SUM(principal), 2) as promo_disbursed_pct,
    
    -- Delinquency KPIs
    ROUND(100.0 * SUM(CASE WHEN max_days_late >= 30 THEN principal ELSE 0 END) / SUM(principal), 2) as PAR30,
    ROUND(100.0 * SUM(CASE WHEN max_days_late >= 60 THEN principal ELSE 0 END) / SUM(principal), 2) as PAR60,
    ROUND(100.0 * SUM(CASE WHEN max_days_late >= 90 THEN principal ELSE 0 END) / SUM(principal), 2) as PAR90
FROM {{ ref('int_loans_repayments') }}
"""

# Write files
with open(f'{base_path}/intermediate/int_loans_repayments.sql', 'w') as f:
    f.write(int_loans_repayments.strip())

with open(f'{base_path}/marts/mart_portfolio_kpis.sql', 'w') as f:
    f.write(mart_portfolio_kpis.strip())

print("Intermediate & Mart models created!")
print("- int_loans_repayments.sql")
print("- mart_portfolio_kpis.sql")

Intermediate & Mart models created!
- int_loans_repayments.sql
- mart_portfolio_kpis.sql


#### Create schema.yml (Documentation + Tests)

In [7]:
schema_yml = """
version: 2

sources:
  - name: mophones_credit
    database: mophones_credit
    schema: mophones_credit
    tables:
      - name: customers
      - name: loans
      - name: repayments

models:
  - name: stg_customers
    description: "Staging table for customer dimension"
    columns:
      - name: customer_id
        tests:
          - unique
          - not_null

  - name: stg_loans
    description: "Staging table for loans fact"
    columns:
      - name: loan_id
        tests:
          - unique
          - not_null
      - name: customer_id
        tests:
          - not_null
          - relationships:
              to: ref('stg_customers')
              field: customer_id

  - name: stg_repayments
    description: "Staging table for repayment transactions"

  - name: int_loans_repayments
    description: "Intermediate table joining loans with repayment performance"

  - name: mart_portfolio_kpis
    description: "Core portfolio health mart with key credit KPIs (PAR30, PAR60, etc.)"
    columns:
      - name: total_disbursed
        description: "Total amount disbursed in KES"
      - name: PAR30
        description: "Portfolio at Risk - 30+ days delinquent"
      - name: PAR60
        description: "Portfolio at Risk - 60+ days delinquent"
"""

# Write the file
with open('../dbt_project/models/schema.yml', 'w') as f:
    f.write(schema_yml.strip())

print("schema.yml created with documentation and tests!")

schema.yml created with documentation and tests!
